# llm based parallel workflow for essay evaluation

In [32]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field


import operator



In [33]:
load_dotenv()  # Load environment variables from .env file
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [34]:
class EvaluateSchema(BaseModel):
    feedback: str = Field(description="detailed feedback for the essay")
    score: int = Field(description="Score out of 10 for the essay", ge=0, le=10)
    

In [35]:
structured_model = model.with_structured_output(EvaluateSchema)

In [36]:
essay = """India in the Age of AI
As the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.

India's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for All) with a focus on inclusive growth, aiming to leverage AI in healthcare, agriculture, education, and smart mobility.

One of the most promising applications of AI in India lies in agriculture, where predictive analytics can guide farmers on optimal sowing times, weather forecasts, and pest control. In healthcare, AI-powered diagnostics can help address India’s doctor-patient ratio crisis, particularly in rural areas. Educational platforms are increasingly using AI to personalize learning paths, while smart governance tools are helping improve public service delivery and fraud detection.

However, the path to AI-led growth is riddled with challenges. Chief among them is the digital divide. While metropolitan cities may embrace AI-driven solutions, rural India continues to struggle with basic internet access and digital literacy. The risk of job displacement due to automation also looms large, especially for low-skilled workers. Without effective skilling and re-skilling programs, AI could exacerbate existing socio-economic inequalities.

Another pressing concern is data privacy and ethics. As AI systems rely heavily on vast datasets, ensuring that personal data is used transparently and responsibly becomes vital. India is still shaping its data protection laws, and in the absence of a strong regulatory framework, AI systems may risk misuse or bias.

To harness AI responsibly, India must adopt a multi-stakeholder approach involving the government, academia, industry, and civil society. Policies should promote open datasets, encourage responsible innovation, and ensure ethical AI practices. There is also a need for international collaboration, particularly with countries leading in AI research, to gain strategic advantage and ensure interoperability in global systems.

India’s demographic dividend, when paired with responsible AI adoption, can unlock massive economic growth, improve governance, and uplift marginalized communities. But this vision will only materialize if AI is seen not merely as a tool for automation, but as an enabler of human-centered development.

In conclusion, India in the age of AI is a story in the making — one of opportunity, responsibility, and transformation. The decisions we make today will not just determine India’s AI trajectory, but also its future as an inclusive, equitable, and innovation-driven society."""

In [37]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
structured_model.invoke(prompt).score

9

In [38]:
structured_model.invoke(prompt).feedback

"The language quality of this essay is exceptionally high. The writing is clear, concise, and professional, making complex ideas easily accessible. The vocabulary is rich and appropriate, demonstrating a strong command of academic and technical terminology without resorting to jargon. Sentence structures are varied and sophisticated, contributing to an engaging and fluid reading experience. There are no noticeable grammatical errors or spelling mistakes, indicating meticulous attention to detail. The essay maintains a consistent, authoritative tone throughout, effectively balancing opportunities with challenges. The transitions between paragraphs are seamless, ensuring a logical flow of arguments. Overall, the linguistic execution significantly enhances the essay's persuasive power and credibility."

In [39]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [40]:
def evaluate_language(state: UPSCState) -> UPSCState:
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)
    
    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}  


In [41]:
def evaluate_thought(state: UPSCState) -> UPSCState:
    prompt = f'Evaluate the clarity and coherence of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)
    
    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [42]:
def evaluate_analysis(state: UPSCState) -> UPSCState:
    prompt = f'Evaluate the analytical quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)
    
    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [43]:
def final_evaluation(state: UPSCState) -> UPSCState:
    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])
    overall_feedback = model.invoke(f"Provide an overall feedback for the essay based on the following feedbacks: {state['language_feedback']}, {state['clarity_feedback']}, {state['analysis_feedback']}")
    
    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [44]:
graph = StateGraph(UPSCState)

# nodes
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

# define edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [45]:
essay2 = """
India in the Age of AI

Artificial Intelligence is very important today and everyone is talking about it. India is also using AI in many places. AI is good because it helps people and makes work easy. Every country wants to become powerful in AI because AI is the future.

India has many engineers and software companies. So India can become number one in AI. Many students are learning AI these days. AI is used in hospitals, schools, farming and offices. It saves time and gives fast results. Therefore AI is useful in every field.

There are some problems also. Some people may lose their jobs because machines can do their work. This is bad for poor people. The government should create more jobs. AI can also make mistakes if people give wrong data. So people should be careful while using AI.

India should invest more money in AI. Every school should teach AI because it is very important. The government should make new rules and everyone should follow them. Companies should also use AI carefully.

In conclusion, AI is good and bad. If India uses AI properly then India will become a developed country. We should use AI for the better future of India because AI is the future and it is useful for everyone.
"""

In [46]:
initial_state = {'essay': essay2}
workflow.invoke(initial_state)

{'essay': '\nIndia in the Age of AI\n\nArtificial Intelligence is very important today and everyone is talking about it. India is also using AI in many places. AI is good because it helps people and makes work easy. Every country wants to become powerful in AI because AI is the future.\n\nIndia has many engineers and software companies. So India can become number one in AI. Many students are learning AI these days. AI is used in hospitals, schools, farming and offices. It saves time and gives fast results. Therefore AI is useful in every field.\n\nThere are some problems also. Some people may lose their jobs because machines can do their work. This is bad for poor people. The government should create more jobs. AI can also make mistakes if people give wrong data. So people should be careful while using AI.\n\nIndia should invest more money in AI. Every school should teach AI because it is very important. The government should make new rules and everyone should follow them. Companies sh